Instalações

In [ ]:
!pip -q install langchain langchain-community langchain-huggingface
!pip -q install pypdf faiss-cpu sentence-transformers transformers accelerate

Upload do PDF

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

print("Arquivo carregado:", pdf_path)

Saving Edital-001_2026.pdf to Edital-001_2026.pdf
Arquivo carregado: Edital-001_2026.pdf


Leitura do PDF


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Total de páginas lidas:", len(documents))
print(documents[0].page_content[:1000])

Total de páginas lidas: 17
Prefeitura de Goiânia
Secretaria Municipal de Administração
Gerência de Recrutamento, Seleção, Promoção e Progressão
EDITAL Nº 001/2026
PROCESSO SELETIVO SIMPLIFICADO PARA SUBSTITUIÇÃO TEMPORÁRIA
SECRETARIA MUNICIPAL DE EDUCAÇÃO
A Secretaria Municipal de Administração de Goiânia, faz saber aos interessados que, nos termos do
art. 37, inciso IX da Constituição Federal; do art. 92, inciso X da Constituição Estadual de Goiás; a Lei
Municipal nº 8.546, de 23 de julho de 2007; do art. 16 da Lei Complementar nº 091, de 26 de junho de 2000
(Estatuto dos Servidores do Magistério Público do Município de Goiânia), da Lei nº 9.128, de 29 de
dezembro de 2011 (Plano de Cargos, Carreiras e Vencimentos dos Trabalhadores Administrativos da
Educação do Município de Goiânia), da Lei nº 7.997, de 20 de junho de 2000 (Plano de Carreira e
Remuneração dos Servidores do Magistério Público do Município de Goiânia) e do Decreto Municipal nº
2.119, de 28 de agosto de 2014, bem como ao

Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,#500,
    chunk_overlap=120#80
)

chunks = splitter.split_documents(documents)

print("Total de chunks gerados:", len(chunks))
print(chunks[0].page_content[:500])

Total de chunks gerados: 128
Prefeitura de Goiânia
Secretaria Municipal de Administração
Gerência de Recrutamento, Seleção, Promoção e Progressão
EDITAL Nº 001/2026
PROCESSO SELETIVO SIMPLIFICADO PARA SUBSTITUIÇÃO TEMPORÁRIA
SECRETARIA MUNICIPAL DE EDUCAÇÃO
A Secretaria Municipal de Administração de Goiânia, faz saber aos interessados que, nos termos do
art. 37, inciso IX da Constituição Federal; do art. 92, inciso X da Constituição Estadual de Goiás; a Lei
Municipal nº 8.546, de 23 de julho de 2007; do art. 16 da Lei Compl


Embeddings + FAISS

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

print("Retriever criado com sucesso.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retriever criado com sucesso.


Busca semântica

In [ ]:
def search_document(question, k=5):
    docs = vectorstore.similarity_search(question, k=k)

    print(f"Pergunta: {question}\n")
    print("Trechos encontrados:\n")

    for i, doc in enumerate(docs, 1):
        print(f"--- Fonte {i} | Página {doc.metadata.get('page', 'N/A')} ---")
        print(doc.page_content[:700])
        print()

    return docs

Contexto formatado

In [ ]:
def build_context(docs):
    partes = []
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get("page", "N/A")
        partes.append(f"[Fonte {i} | Página {page}]\n{doc.page_content}")
    return "\n\n".join(partes)

Resposta guiada


In [ ]:
def answer_with_context(question, k=5):
    docs = vectorstore.similarity_search(question, k=k)
    context = build_context(docs)

    print("Pergunta:")
    print(question)
    print("\nResposta guiada:")
    print("Os trechos abaixo foram recuperados como os mais relevantes para responder à pergunta.")
    print("A interpretação final pode ser feita sobre essas fontes.")
    print("\nContexto recuperado:\n")
    print(context[:4000])

    return docs, context

Extração simples de datas

In [ ]:
import re

def extract_dates(question, k=5):
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n".join([doc.page_content for doc in docs])

    datas = re.findall(r'\d{2}/\d{2}/\d{4}(?:\s*às\s*\d{1,2}h\d{0,2}min)?', context)
    datas_unicas = sorted(set(datas))

    print("Pergunta:", question)
    print("\nDatas encontradas:")
    if datas_unicas:
        for d in datas_unicas:
            print("-", d)
    else:
        print("Nenhuma data encontrada nos trechos recuperados.")

    print("\nFontes:")
    for i, doc in enumerate(docs, 1):
        print(f"\n--- Fonte {i} | Página {doc.metadata.get('page', 'N/A')} ---")
        print(doc.page_content[:500])

Extração de itens em lista

In [ ]:
def extract_list_items(question, k=5):
    docs = vectorstore.similarity_search(question, k=k)

    print("Pergunta:", question)
    print("\nItens encontrados:\n")

    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get("page", "N/A")
        print(f"--- Fonte {i} | Página {page} ---")

        linhas = doc.page_content.split("\n")
        itens = [linha.strip() for linha in linhas if linha.strip().startswith(("a)", "b)", "c)", "d)", "e)", "f)", "g)", "h)", "i)", "j)", "k)", "l)", "m)", "n)", "o)", "p)", "q)", "r)", "s)", "t)"))]

        if itens:
            for item in itens:
                print(item)
        else:
            print(doc.page_content[:700])

        print()

# **Exemplos de uso**

In [ ]:
extract_dates("Qual é o período de inscrição?")

Pergunta: Qual é o período de inscrição?

Datas encontradas:
- 06/04/2026 às 23h59min

Fontes:

--- Fonte 1 | Página 3 ---
nome, número dos documentos de identiﬁ cação e data de nascimento.
3.7 Em caso de erros nos dados de inscrição e/ou nos dados relativos a Títulos e Experiência
Proﬁ ssional inseridos para ﬁns de pontuação, o candidato terá até o último dia do período de inscrição,
ou seja, dia 06/04/2026 às 23h59min para corrigi-los.
3.7.1 Após esta data não serão aceitos, em nenhuma hipótese, acréscimos, complementações ou
alterações nos referidos dados de inscrição.
3.7.2 No ato da inscrição o candidato não po

--- Fonte 2 | Página 9 ---
a) Em Órgão Público: Declaração ou Certidão de Tempo de Serviço constando a data da posse e da
exoneração (se for o caso), a função exercida e a descrição das atividades desenvolvidas, emitida em papel
timbrado com carimbo do órgão expedidor, datado e assinado pelo Departamento de Pessoal / Recursos
Humanos do órgão onde prestou serviço, não send

In [ ]:
extract_list_items("Quais são os documentos exigidos para contratação?")

Pergunta: Quais são os documentos exigidos para contratação?

Itens encontrados:

--- Fonte 1 | Página 13 ---
a) não atender os requisitos necessários para a função (Anexo II);
b) não apresentar a documentação comprobatória indicada na Avaliação de Títulos e Experiência

--- Fonte 2 | Página 13 ---
substituição aos documentos exigidos.
13.26 Os documentos e requisitos descritos no item 13.22 deste Edital deverão ser atendidos
cumulativamente, sendo avaliados para, posteriormente, ocorrer a assinatura do contrato.
13.27 Todos os documentos comprobatórios deverão ser apresentados no ato da convocação e
contratação;
13.28 Toda a documentação será avaliada pela equipe técnica da Secretaria Municipal de Educação
que, após análise, efetivará a contratação.
13.29 Os candidatos com de ﬁciência, além dos documentos exigidos para contratação, deverão
apresentar os documentos descritos no item 5.2.1.
13.30 Os candidatos que se autodeclararem negros, além dos documentos exigidos para

--- Fonte 3 

In [ ]:
extract_list_items("Quais são os requisitos para contratação temporária?")

Pergunta: Quais são os requisitos para contratação temporária?

Itens encontrados:

--- Fonte 1 | Página 13 ---
a) Cumprir as determinações do presente Edital;
b) Não possuir contrato temporário com este Município que se enquadre nos termos do art. 7º da

--- Fonte 2 | Página 15 ---
18.2 É vedada a recontratação do pessoal admitido nos termos desta Lei nº 8.546/2007, na mesma
ou em outra função, antes de decorridos 12 (doze) meses, contados do término do contrato, exceto, para as
situações previstas nos incisos I e II do art.7º da referida lei.
18.3 A classiﬁcação neste Processo Seletivo Simpliﬁcado gera apenas a expectativa de direito a
contratação temporária. É reservado à Secretaria Municipal de Educação o direito de proceder à
contratação temporária em número que atenda aos seus interesses e às suas necessidades.
18.4 Será eliminado do Processo Seletivo o candidato que ﬁzer declaração falsa ou inexata em

--- Fonte 3 | Página 15 ---
oﬁcial do certame, referente aos resultados, recu

In [ ]:
print("="*60)
search_document("Em quais situações o candidato pode ser eliminado do processo seletivo?")

Pergunta: Em quais situações o candidato pode ser eliminado do processo seletivo?

Trechos encontrados:

--- Fonte 1 | Página 3 ---
distintos sem sobreposição de tempo.
3.8 O descumprimento de quaisquer das instruções deste Edital para inscrição via Internet implicará
no cancelamento da mesma.
3.9 A qualquer tempo poderá ser anulada a inscrição dos candidatos que prestarem declaração falsa
ou inexata no ato da inscrição, ou apresentarem irregularidades nas provas documentais no momento da
convocação do candidato.
3.10 Após a inscrição o candidato interessado em realizar a contratação temporária poderá imprimir,
consultar e acompanhar a sua inscrição no site www.goiania.go.gov.br, na página reservada aos Concursos e
Seleções, no link “Processo Seletivo Simpliﬁ

--- Fonte 2 | Página 3 ---
da elegibilidade e da aptidão do candidato, regras de desempate, em conformidade com a legislação
aplicável, visando a seleção e formalização de um eventual contrato temporário com o Município.
3.11.2 O

[Document(id='dce0a685-ef54-4a9f-841b-dfb9d9435724', metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260314103508', 'source': 'Edital-001_2026.pdf', 'total_pages': 17, 'page': 3, 'page_label': '4'}, page_content='distintos sem sobreposição de tempo.\n3.8 O descumprimento de quaisquer das instruções deste Edital para inscrição via Internet implicará\nno cancelamento da mesma.\n3.9 A qualquer tempo poderá ser anulada a inscrição dos candidatos que prestarem declaração falsa\nou inexata no ato da inscrição, ou apresentarem irregularidades nas provas documentais no momento da\nconvocação do candidato.\n3.10 Após a inscrição o candidato interessado em realizar a contratação temporária poderá imprimir,\nconsultar e acompanhar a sua inscrição no site www.goiania.go.gov.br, na página reservada aos Concursos e\nSeleções, no link “Processo Seletivo Simpliﬁ'),
 Document(id='ff67a6fb-ad68-4a91-82aa-4d2171ebc311', metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creat